# KL vs VQ First-Stage Autoencoder Comparison
## Experiment 1: No_Mixed Dataset

This notebook trains both KL and VQ autoencoders on the No_Mixed dataset and compares their reconstruction quality and boundary preservation.

**Kaggle Setup:**
1. Enable GPU (T4 or P100)
2. Add the No_Mixed dataset to the notebook
3. Clone/pull the Mixed-Pixels repository
4. Run all cells

## 1. Environment Setup

In [ ]:
import os
import sys
from pathlib import Path

# Kaggle paths
KAGGLE_INPUT = Path('/kaggle/input')
KAGGLE_WORKING = Path('/kaggle/working')

# Check if running on Kaggle
IS_KAGGLE = KAGGLE_INPUT.exists()

print(f"Running on Kaggle: {IS_KAGGLE}")
print(f"Working directory: {os.getcwd()}")

if IS_KAGGLE:
    print(f"\nAvailable datasets in /kaggle/input:")
    if KAGGLE_INPUT.exists():
        for item in KAGGLE_INPUT.iterdir():
            print(f"  - {item.name}")

In [ ]:
# Clone or navigate to repository
if IS_KAGGLE:
    REPO_NAME = 'mixed-pixels'  # Adjust to your repo name
    
    if not (KAGGLE_WORKING / REPO_NAME).exists():
        print("Cloning repository...")
        # If you've already added the repo as a dataset:
        # !cp -r /kaggle/input/mixed-pixels-repo/* /kaggle/working/
        
        # Or clone from GitHub:
        !git clone https://github.com/YOUR_USERNAME/mixed-pixels.git
        
    os.chdir(KAGGLE_WORKING / REPO_NAME)
    print(f"Changed to: {os.getcwd()}")
    
    # Add src to path
    sys.path.insert(0, str(Path.cwd() / 'src'))
else:
    # Local development
    repo_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    sys.path.insert(0, str(repo_root / 'src'))

print(f"\nPython path includes: {sys.path[0]}")

In [ ]:
# Check GPU availability
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 2. Configure Paths for Kaggle

In [ ]:
import yaml
from pathlib import Path

# Load base configurations
config_dir = Path('configs/experiment1')
kl_config_path = config_dir / 'kl_autoencoder.yaml'
vq_config_path = config_dir / 'vq_autoencoder.yaml'

with open(kl_config_path, 'r') as f:
    kl_config = yaml.safe_load(f)

with open(vq_config_path, 'r') as f:
    vq_config = yaml.safe_load(f)

# Override paths for Kaggle
if IS_KAGGLE:
    # Adjust dataset path - update 'no-mixed-dataset' to match your Kaggle dataset name
    DATASET_NAME = 'no-mixed-dataset'  # CHANGE THIS to your actual dataset name
    dataset_root = KAGGLE_INPUT / DATASET_NAME
    
    if not dataset_root.exists():
        print(f"WARNING: Dataset not found at {dataset_root}")
        print(f"Available inputs: {list(KAGGLE_INPUT.iterdir())}")
    
    # Update both configs
    for config in [kl_config, vq_config]:
        config['data']['dataset_root'] = str(dataset_root)
        
        # Output paths in Kaggle working directory
        exp_name = config['experiment']['name']
        config['paths']['checkpoint_dir'] = str(KAGGLE_WORKING / 'checkpoints' / exp_name)
        config['paths']['log_dir'] = str(KAGGLE_WORKING / 'logs' / exp_name)
        config['paths']['tensorboard_dir'] = str(KAGGLE_WORKING / 'runs' / exp_name)
        config['paths']['output_dir'] = str(KAGGLE_WORKING / 'outputs' / exp_name)
        
        # Ensure directories exist
        for path_key in ['checkpoint_dir', 'log_dir', 'tensorboard_dir', 'output_dir']:
            Path(config['paths'][path_key]).mkdir(parents=True, exist_ok=True)

print("KL Config - Dataset root:", kl_config['data']['dataset_root'])
print("KL Config - Checkpoint dir:", kl_config['paths']['checkpoint_dir'])
print("\nVQ Config - Dataset root:", vq_config['data']['dataset_root'])
print("VQ Config - Checkpoint dir:", vq_config['paths']['checkpoint_dir'])

## 3. Verify Dataset

In [ ]:
# Check dataset structure
dataset_path = Path(kl_config['data']['dataset_root'])

print(f"Dataset path: {dataset_path}")
print(f"Dataset exists: {dataset_path.exists()}")

if dataset_path.exists():
    # List contents
    items = list(dataset_path.iterdir())
    print(f"\nDataset contains {len(items)} items")
    
    # Count image files
    image_extensions = {'.png', '.jpg', '.jpeg', '.tif', '.tiff'}
    images = [f for f in dataset_path.rglob('*') if f.suffix.lower() in image_extensions]
    print(f"Total images found: {len(images)}")
    
    if images:
        print(f"\nFirst few images:")
        for img in images[:5]:
            print(f"  - {img.relative_to(dataset_path)}")
else:
    print("\n⚠️ ERROR: Dataset not found!")
    print("Please ensure the No_Mixed dataset is added to this notebook.")

## 4. Import Training Modules

In [ ]:
# Import project modules
try:
    # Import training script and evaluation
    from scripts.training.train_first_stage import train_first_stage
    from scripts.evaluation.evaluate_first_stage import evaluate_first_stage
    
    # Import model classes for reference
    from models.ldm.first_stage.kl.model import KLAutoencoder
    from models.ldm.first_stage.vq.model import VQAutoencoder
    
    print("✓ Successfully imported training modules")
except ImportError as e:
    print(f"✗ Import error: {e}")
    print("\nChecking src directory structure...")
    src_path = Path('src')
    if src_path.exists():
        print("\nKey directories:")
        for item in ['models/ldm/first_stage', 'scripts/training', 'scripts/evaluation']:
            full_path = src_path / item
            if full_path.exists():
                print(f"  ✓ {item}")
                for f in full_path.glob('*.py'):
                    print(f"    - {f.name}")
            else:
                print(f"  ✗ {item} not found")

## 5. Train KL Autoencoder

In [ ]:
print("="*60)
print("Training KL-Regularized Autoencoder")
print("="*60)

# Train KL model
kl_results = train_first_stage(
    config=kl_config,
    model_type='kl'
)

print("\n" + "="*60)
print("KL Training Complete")
print("="*60)
print(f"Final validation loss: {kl_results.get('final_val_loss', 'N/A')}")
print(f"Best epoch: {kl_results.get('best_epoch', 'N/A')}")
print(f"Checkpoints saved to: {kl_config['paths']['checkpoint_dir']}")

## 6. Train VQ Autoencoder

In [ ]:
print("="*60)
print("Training VQ-Regularized Autoencoder")
print("="*60)

# Train VQ model
vq_results = train_first_stage(
    config=vq_config,
    model_type='vq'
)

print("\n" + "="*60)
print("VQ Training Complete")
print("="*60)
print(f"Final validation loss: {vq_results.get('final_val_loss', 'N/A')}")
print(f"Best epoch: {vq_results.get('best_epoch', 'N/A')}")
print(f"Final codebook utilization: {vq_results.get('final_codebook_utilization', 'N/A')}")
print(f"Checkpoints saved to: {vq_config['paths']['checkpoint_dir']}")

## 7. Evaluate Both Models

In [ ]:
print("="*60)
print("Evaluating KL Autoencoder")
print("="*60)

# Find best KL checkpoint
kl_checkpoint_dir = Path(kl_config['paths']['checkpoint_dir'])
kl_best_ckpt = kl_checkpoint_dir / 'best_model.pth'

if kl_best_ckpt.exists():
    kl_eval_results = evaluate_first_stage(
        config=kl_config,
        checkpoint_path=str(kl_best_ckpt),
        model_type='kl'
    )
    print("\nKL Evaluation Results:")
    for metric, value in kl_eval_results.items():
        print(f"  {metric}: {value}")
else:
    print(f"Warning: Best checkpoint not found at {kl_best_ckpt}")

In [ ]:
print("="*60)
print("Evaluating VQ Autoencoder")
print("="*60)

# Find best VQ checkpoint
vq_checkpoint_dir = Path(vq_config['paths']['checkpoint_dir'])
vq_best_ckpt = vq_checkpoint_dir / 'best_model.pth'

if vq_best_ckpt.exists():
    vq_eval_results = evaluate_first_stage(
        config=vq_config,
        checkpoint_path=str(vq_best_ckpt),
        model_type='vq'
    )
    print("\nVQ Evaluation Results:")
    for metric, value in vq_eval_results.items():
        print(f"  {metric}: {value}")
else:
    print(f"Warning: Best checkpoint not found at {vq_best_ckpt}")

## 8. Compare Results

In [ ]:
import pandas as pd

# Create comparison table
comparison = {
    'Metric': [],
    'KL': [],
    'VQ': [],
    'Difference (KL - VQ)': []
}

# Common metrics
common_metrics = ['mse', 'mae', 'psnr', 'ssim', 
                  'boundary_mse', 'non_boundary_mse', 'boundary_ratio']

for metric in common_metrics:
    if metric in kl_eval_results and metric in vq_eval_results:
        kl_val = kl_eval_results[metric]
        vq_val = vq_eval_results[metric]
        comparison['Metric'].append(metric)
        comparison['KL'].append(f"{kl_val:.6f}")
        comparison['VQ'].append(f"{vq_val:.6f}")
        comparison['Difference (KL - VQ)'].append(f"{kl_val - vq_val:.6f}")

df = pd.DataFrame(comparison)
print("\n" + "="*60)
print("KL vs VQ Comparison")
print("="*60)
print(df.to_string(index=False))

# Save comparison
comparison_path = KAGGLE_WORKING / 'kl_vs_vq_comparison.csv' if IS_KAGGLE else Path('outputs/kl_vs_vq_comparison.csv')
df.to_csv(comparison_path, index=False)
print(f"\nComparison saved to: {comparison_path}")

## 9. Visualize Sample Reconstructions

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

# Load sample reconstructions from output directories
kl_output = Path(kl_config['paths']['output_dir'])
vq_output = Path(vq_config['paths']['output_dir'])

# Check for saved visualizations
kl_recon_files = list(kl_output.glob('*reconstruction*.png'))
vq_recon_files = list(vq_output.glob('*reconstruction*.png'))

if kl_recon_files and vq_recon_files:
    # Display first few reconstructions
    num_samples = min(3, len(kl_recon_files), len(vq_recon_files))
    
    fig, axes = plt.subplots(num_samples, 2, figsize=(12, 4*num_samples))
    if num_samples == 1:
        axes = axes.reshape(1, -1)
    
    for i in range(num_samples):
        # KL
        kl_img = Image.open(kl_recon_files[i])
        axes[i, 0].imshow(kl_img)
        axes[i, 0].set_title(f'KL Reconstruction {i+1}')
        axes[i, 0].axis('off')
        
        # VQ
        vq_img = Image.open(vq_recon_files[i])
        axes[i, 1].imshow(vq_img)
        axes[i, 1].set_title(f'VQ Reconstruction {i+1}')
        axes[i, 1].axis('off')
    
    plt.tight_layout()
    vis_save_path = KAGGLE_WORKING / 'reconstruction_comparison.png' if IS_KAGGLE else Path('outputs/reconstruction_comparison.png')
    plt.savefig(vis_save_path, dpi=150, bbox_inches='tight')
    print(f"Visualization saved to: {vis_save_path}")
    plt.show()
else:
    print("No reconstruction visualizations found.")
    print(f"KL output dir: {kl_output} (exists: {kl_output.exists()})")
    print(f"VQ output dir: {vq_output} (exists: {vq_output.exists()})")

## 10. Summary and Next Steps

In [ ]:
print("="*60)
print("Experiment Complete")
print("="*60)

print("\n📊 Results Summary:")
print(f"  - KL checkpoints: {kl_config['paths']['checkpoint_dir']}")
print(f"  - VQ checkpoints: {vq_config['paths']['checkpoint_dir']}")
print(f"  - Comparison table: {comparison_path}")

print("\n📥 To download results:")
print("  1. Check the Output tab in Kaggle")
print("  2. Download checkpoints/, outputs/, and comparison CSV")
print("  3. Add results to your local repository")

print("\n📝 Next Steps:")
print("  1. Analyze the comparison metrics")
print("  2. Examine reconstruction quality visually")
print("  3. Compare boundary preservation (boundary_ratio)")
print("  4. Update docs/CURRENT_STATE.md with findings")
print("  5. Decide on next experiment based on results")